In [2]:
import os
import sys
import time
import json
from dotenv import load_dotenv

load_dotenv()
import logging

# logging.basicConfig(level=logging.DEBUG, stream=sys.stdout)
from pydantic import BaseModel
import importlib

import agents

importlib.reload(agents)

from agents.templates.play_zero_agent import PlayZeroAgent
from agents.structs import FrameData, GameState

import textwrap

import json
from pydantic import BaseModel, Field
from typing import List

EVAL_MAPPING_FILE_PATH = "data/eval/eval_data_input_mapping.json"
with open(EVAL_MAPPING_FILE_PATH, "r") as f:
    eval_data_input_mapping = json.load(f)

def print_wrapped_text(text: str, width: int = 80):
    """
    Print the given text with word-wrapped lines for better readability in the terminal.

    Args:
        text (str): The input text to be printed.
        width (int): The maximum line width before wrapping. Default is 80.
    """
    wrapper = textwrap.TextWrapper(width=width)
    paragraphs = text.strip().split("\n\n")

    for paragraph in paragraphs:
        wrapped = wrapper.fill(paragraph)
        print(wrapped + "\n")

#  Agent.__init__() missing 5 required positional arguments: 'card_id', 'game_id', 'agent_name', 'ROOT_URL', and 'record'
play_zero_agent: PlayZeroAgent = PlayZeroAgent(
    card_id="play_zero_agent",
    game_id="play_zero_agent",
    agent_name="PlayZeroAgent",
    ROOT_URL="http://localhost:8000",
    record=False,
)

runs = [
    {
        "game_id": "ls20",
        "level": 1,
        "video_path": "/workspaces/ARC-AGI-3-Agents/recordings/game_analysis_ls20-f340c8e5138e_track1.mp4",
        "scorecard_file_path": "/workspaces/ARC-AGI-3-Agents/recordings/ls20-f340c8e5138e.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.8aefc1ea-8e65-41ec-9272-a8faff24ddb1.recording.jsonl",
        "frame_start": 0,
        "frame_end": 55,
        "expected_goal": "You need to move the \"Orange-Capped Blue Block (6x7)\" to the target \"8x7Grid_BlackHead_BlueEye_WhiteSnout\"",
    },
    {
        "game_id": "ls20",
        "level": 1,
        "video_path": "/workspaces/ARC-AGI-3-Agents/recordings/game_analysis_ls20-f340c8e5138e_track2.mp4",
        "scorecard_file_path": "/workspaces/ARC-AGI-3-Agents/recordings/ls20-f340c8e5138e.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.fa1f97de-edb6-47c7-98bb-eff9f689d487.recording.jsonl",  
        "frame_start": 0,
        "frame_end": 55,
        "expected_goal": "You need to move the \"Orange-Capped Blue Block (6x7)\" to the target \"8x7Grid_BlackHead_BlueEye_WhiteSnout\"",
    },
    {
        "game_id": "vc33",
        "level": 1,
        "video_path": "/workspaces/ARC-AGI-3-Agents/recordings/game_analysis_vc33-58ec4396715d_track1.mp4",
        "scorecard_file_path": "recordings/vc33-58ec4396715d.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.5fe87248-8898-45b1-8634-84294454be48.recording.jsonl",
        "frame_start": 0,
        "frame_end": 55,
        "expected_goal": "Click red button and blue button and notice the effect",
    },
    {
        "game_id": "vc33",
        "level": 1,
        "video_path": "",
        "scorecard_file_path": "recordings/vc33-58ec4396715d.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.28b07367-701c-470f-93d6-b302ecbbc733.recording.jsonl",
        "frame_start": 0,
        "frame_end": 55,
        "expected_goal": "Click red button and blue button and notice the effect",
    }
]

def get_frames(scorecard_file_path):
    with open(scorecard_file_path, "r") as file:
        grid_jsons = [json.loads(line) for line in file]
    frames = [FrameData(**frame_json["data"]) for frame_json in grid_jsons]
    frames.insert(0, FrameData(score=0))
    return frames

def write_eval_log(evaluation_result):
    with open("eval.log", "a") as eval_log_file:
        print_wrapped_text(f"Evaluation Result:\n\n {evaluation_result}\n")
        eval_log_file.write(f"Evaluation Result:\n\n {evaluation_result}\n")

# Eval prompt for multiple_hypothesis_text

EVAL_PROMPT = """Give score and reason of whether the multiple hypothesis can be used to generate the expected goal

Expected Goal: <expected_goal>{expected_goal}</expected_goal>

Multiple Hypothesis Text: <multiple_hypothesis_text>{multiple_hypothesis_text}</multiple_hypothesis_text>

Example output json:
```json
{{
    "reason": "<max of 100 words>",
    "score": "<float score from 0.0 to 1.0>"
}}
```
"""

GOAL_RELEVANCE_PROMPT = """Give score and reason of whether the generated goal can be used to achieve the expected goal

Expected Goal: <expected_goal>{expected_goal}</expected_goal>

Generated Goal: <generated_goal>{generated_goal}</generated_goal>

Example output json:
```json
{{
    "reason": "<max of 100 words>",
    "score": "<float score from 0.0 to 1.0>"
}}
```
"""


def evaluate_multiple_hypothesis_text(multiple_hypothesis_text: str, expected_goal: str):
    prompt = EVAL_PROMPT.format(
        expected_goal=expected_goal,
        multiple_hypothesis_text=multiple_hypothesis_text
    )
    response = play_zero_agent.client.chat.completions.create(
        model="gemini-2.5-flash",
        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    json_text = response.choices[0].message.content.strip()
    try:
        json_text_extracted = play_zero_agent.extract_first_json_block(json_text)
        json_data = json.loads(json_text_extracted)
    except json.JSONDecodeError:
        json_data = {
            "reason": json_text,
            "score": 0,
        }
    return json_data

def eval_goal_relevance(generated_goal: str, expected_goal: str):
    prompt = GOAL_RELEVANCE_PROMPT.format(
        expected_goal=expected_goal,
        generated_goal=generated_goal
    )
    response = play_zero_agent.client.chat.completions.create(
        model="gemini-2.5-flash",
        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    json_text = response.choices[0].message.content.strip()
    try:
        json_text_extracted = play_zero_agent.extract_first_json_block(json_text)
        json_data = json.loads(json_text_extracted)
    except json.JSONDecodeError:
        json_data = {
            "reason": json_text,
            "score": 0,
        }
    return json_data

class RunResult(BaseModel):
    logical_analysis_actions_summary: str = ""
    eval_multiple_hypothesis_result: dict = {}
    eval_goal_relevance_result: dict = {}
    multiple_hypothesis_text: str = ""
    goal: str = ""
    elements_text: str = ""

def store_run_results(run_results: list[dict]):
    # generate a unique path for the run results file
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_results_path = f"data/run_results_{timestamp}.json"
    with open(run_results_path, "w") as file:
        json.dump(run_results, file)

class EvalDataItem(BaseModel):
    game_id: str = Field(..., description="Unique game identifier, e.g., 'vc33'")
    game_name: str = Field(..., description="Name of the game, e.g., 'VALVECHECK'")
    level: int = Field(..., description="Level number of the game")
    video_path: str = Field(..., description="Path to the MP4 game analysis video")
    scorecard_file_path: str = Field(..., description="Path to the scorecard JSONL file")
    frame_start: int = Field(..., description="Starting frame number in the video")
    frame_end: int = Field(..., description="Ending frame number in the video")
    expected_goal: str = Field(..., description="Description of the expected game goal")

eval_data_list = [
    EvalDataItem(**sample) for sample in eval_data_input_mapping
]

/workspaces/ARC-AGI-3-Agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 

In [4]:

# def test_run(eval_data: EvalDataItem) -> RunResult:
#     video_path = eval_data.video_path
#     scorecard_file_path = eval_data.scorecard_file_path
#     frame_start = eval_data.frame_start
#     frame_end = eval_data.frame_end
#     expected_goal = eval_data.expected_goal

#     frames = get_frames(scorecard_file_path)
#     frames_for_analysis = frames[frame_start:frame_end] if frame_end else frames[frame_start:]

#     print(f"Running analysis for video: {video_path}")
#     print(f"Scorecard file: {scorecard_file_path}")
#     print(f"Frames from {frame_start} to {frame_end if frame_end else 'end'}")

#     logical_analysis_actions_summary = play_zero_agent.generate_logical_analysis_summary(frames_for_analysis)
#     print(f"Logical Analysis Actions Summary:\n\n {logical_analysis_actions_summary}")
#     # elements_text = play_zero_agent.generate_element_titles_from_video(
#     #     video_file_path=video_path,
#     # )
#     # print_wrapped_text(f"elements_text:\n\n {elements_text}")
#     effective_frames = play_zero_agent.generate_video_from_grids(
#         frames_for_analysis,
#         video_output_path=video_path,
#         fps=1,
#         skip_repeated_frames=True,
#     )
#     generate_list_of_actions(effective_frames)

#     multiple_hypothesis_text = play_zero_agent.generate_multiple_random_hypotheses_from_video(
#         video_file_path=video_path,
#         logical_analysis_actions_summary=logical_analysis_actions_summary,
#     )
#     print_wrapped_text(f"Multiple Hypothesis Text:\n\n {multiple_hypothesis_text}")
#     eval_multiple_hypothesis_result = evaluate_multiple_hypothesis_text(
#         multiple_hypothesis_text=multiple_hypothesis_text,
#         expected_goal=expected_goal,
#     )
#     write_eval_log(eval_multiple_hypothesis_result)
#     goal = play_zero_agent.generate_top_hypothesis(
#         multiple_hypothesis_text=multiple_hypothesis_text,
#         logical_analysis_actions_summary=logical_analysis_actions_summary,
#     )
#     print_wrapped_text(f"Generated Goal:\n\n {goal}")
#     eval_goal_relevance_result = eval_goal_relevance(
#         generated_goal=goal,
#         expected_goal=expected_goal
#     )
#     write_eval_log(eval_goal_relevance_result)


#     return RunResult(
#         logical_analysis_actions_summary=logical_analysis_actions_summary,
#         multiple_hypothesis_text=multiple_hypothesis_text,
#         eval_multiple_hypothesis_result=eval_multiple_hypothesis_result,
#         eval_goal_relevance_result=eval_goal_relevance_result,
#         goal=goal,
#         elements_text=""
#     )

# run_results = []
# for eval_data in eval_data_list:
#     run_result = test_run(eval_data)
#     run_results.append({
#         "input": eval_data,
#         "result": run_result.model_dump()
#     })
# store_run_results(run_results)

In [5]:
"""The two images are frames of  ValveCheck game. THe first frame is past frame and a clicked (60,28) blue cell position action in the first frame of game image. The second frame has the effects of the action. 

Some logical changes in cell.

- 100 grey cells turned to white on left
- 100 white cells turned to grey on right
- 24 white cells turned to yellow on right
- 24 yellow cells turned to grey on right

Can you find the analogy why these cells in the game background and objects changed. There can be minor changes in background as well.
"""

'The two images are frames of  ValveCheck game. THe first frame is past frame and a clicked (60,28) blue cell position action in the first frame of game image. The second frame has the effects of the action. \n\nSome logical changes in cell.\n\n- 100 grey cells turned to white on left\n- 100 white cells turned to grey on right\n- 24 white cells turned to yellow on right\n- 24 yellow cells turned to grey on right\n\nCan you find the analogy why these cells in the game background and objects changed. There can be minor changes in background as well.\n'

In [14]:
from collections import Counter

def summarize_frame_diff(self, previous_frame: FrameData, current_frame: FrameData) -> list[str]:
    changes = Counter()
    prev_grid = previous_frame.frame[-1]
    curr_grid = current_frame.frame[0]

    rows = len(prev_grid)
    cols = len(prev_grid[0])
    mid_col = cols // 2  # split left/right halves

    for r in range(rows):
        for c in range(cols):
            old_val = prev_grid[r][c]
            new_val = curr_grid[r][c]
            if old_val != new_val:
                old_color = self.get_color_for_cell_value(old_val)
                new_color = self.get_color_for_cell_value(new_val)
                side = "left" if c < mid_col else "right"
                changes[(old_color, new_color, side)] += 1

    # Turn counts into readable text
    summary_lines = [
        f"- {count} {old} cells turned to {new} on {side}"
        for (old, new, side), count in changes.items()
    ]
    return summary_lines

def generate_list_of_actions(
    frames: list[FrameData],
) -> str:
    actions = []
    for i, frame in enumerate(frames[1:]):
        actions_text = "RESET"
        if frame.action_input.reasoning:
            actions_text = frame.action_input.reasoning.get("previous_action_text", actions_text)
        actions.append(actions_text)
    return "\n- ".join(actions)

logger = logging.getLogger(__name__)
def generate_event_chain(self: PlayZeroAgent, effective_frames: list[FrameData]) -> str:
    count = 0
    event_chain = []
    prev_frame = effective_frames[0]
    for frame in effective_frames:
        if not frame.action_input.reasoning:
            continue
        game_action = frame.action_input.id
        frame_count = len(frame.frame)
        if game_action.is_complex():
            print(game_action.action_data.x)
            x = frame.action_input.data["x"]
            y = frame.action_input.data["y"]
            game_action.set_data(
                {
                    "x": x,
                    "y": y,
                }
            )
        
            cell_value = prev_frame.frame[-1][y][x]
            color = self.get_color_for_cell_value(cell_value)
            if color:
                frame_event = self.convert_game_action_to_text(game_action)
                notes = f"{frame_event}{color} cell. This has effect in game"
        else:
            action_text = self.convert_game_action_to_text(game_action)
            notes = f"Taking {action_text} action"
        event_chain.append(notes)
        count += frame_count
    return "\n".join(event_chain)


eval_data = eval_data_list[0]
frames = get_frames(eval_data.scorecard_file_path)
effective_frames = play_zero_agent.generate_video_from_grids(frames, "output.mp4", fps=1, skip_repeated_frames=True)

diff = summarize_frame_diff(play_zero_agent, previous_frame=effective_frames[0], current_frame=effective_frames[1])
print("Frame Diff Summary:")
print("\n".join(diff))
actions_text = generate_list_of_actions(effective_frames)
print(f"Len of actual frames: {len(frames)}")
print(f"len of effective frames: {len(effective_frames)}")
print(effective_frames[0].action_input)
event_chain = generate_event_chain(play_zero_agent, effective_frames)
print(f"Actions text: {event_chain}")

Frame Diff Summary:
- 1 Bright Green cells turned to White on right
- 96 White cells turned to Medium Gray on left
- 32 Medium Gray cells turned to Bright Yellow on right
- 16 Medium Gray cells turned to White on left
- 96 Medium Gray cells turned to White on right
- 32 Bright Yellow cells turned to White on right
Len of actual frames: 265
len of effective frames: 56
id=<GameAction.RESET: 0> data={'game_id': 'vc33-58ec4396715d'} reasoning={'desired_action': '0', 'elements_text': 'No elements description available yet.', 'goal': '', 'goal_actions_count': 0, 'hints': 'No hints available yet.', 'logical_analysis_actions_summary': 'No logical analysis actions summary available yet.', 'max_goal_actions_limit': 30, 'multiple_hypothesis_text': 'No hypotheses available yet.', 'previous_action_reason': 'Game has not been played yet, resetting.', 'previous_action_text': 'RESET', 'reason': 'Game has not been played yet, resetting.'}
4
28
28
20
12
12
20
12
12
4
28
12
36
28
12
4
4
36
28
12
36
36
12

In [32]:
# from google.genai import types

# logger = logging.getLogger(__name__)

# RANDOM_GAME_NAME_PROMPT = """This video is game play with the below actions (WASD and click) taken on unknown game.

# The game is designed based on below Constraints
# - Easy for humans (can pick it up in <1 min of game play)
# - Core Knowledge Priors (no language, trivia, cultural symbols)
# - Should require no instructions to play
# - Should be fun for humans and playable in 5-10 minutes
# - Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)

# Some hypothesis generated by human when he played the game

# Here are 5 hypotheses to explore the game:\n\n1.  **Core Objective:** The primary objective of the game is to maneuver the `Blue_Square_Movable_Block` and its `Orange_Rectangle_Movable_Cap` across the `Irregular_Shape_Light_Grey_Playfield` so that the `Orange_Rectangle_Movable_Cap` is successfully placed directly onto the `Black_Square_Goal_Block_with_Blue_Dot`.\n2.  **Player Control and Interaction:** The `White_L-Shaped_Player_Block` is the player's avatar, which moves within the `Irregular_Shape_Light_Grey_Playfield` using WASD inputs. It can push the `Blue_Square_Movable_Block`, and the `Orange_Rectangle_Movable_Cap` remains on top of and moves with the `Blue_Square_Movable_Block` as it is pushed.\n3.  **Failure Condition:** A life is lost, indicated by one of the `Red_Square_Life_Indicators` turning into a `Grey_Square_Life_Indicators` and a temporary `Red_Game_Over_Background` screen, when the `Orange_Rectangle_Movable_Cap` is separated from the `Blue_Square_Movable_Block` or pushed beyond the intended target area of the `Black_Square_Goal_Block_with_Blue_Dot` and `White_Inverted_T-Shaped_Goal_Base`.\n4.  **Progress Tracking:** The `Purple_Square_Progress_Indicators` at the top of the screen track progress, converting to `Grey_Square_Progress_Indicators` when a level or sub-objective is successfully completed by moving the `Orange_Rectangle_Movable_Cap` to the `Black_Square_Goal_Block_with_Blue_Dot`.\n5.  **Environmental Obstacles:** The `White_T-Shaped_Static_Block` and the `Dark_Grey_Vertical_Wall_Block` are fixed obstacles that restrict the movement paths of the `White_L-Shaped_Player_Block`, the `Blue_Square_Movable_Block`, and the `Orange_Rectangle_Movable_Cap`.

# Can you give me decription of game IN 1 LINE using core game mechanics?
# """

# def generate_game_name_from_video(
#     self,
#     video_file_path: str,
# ) -> str:
#     """Generate a game name from a video file."""
#     logger.info(f"Generating game name from video: {video_file_path}")
#     if not os.path.exists(video_file_path):
#         logger.error(f"Video file does not exist: {video_file_path}")
#         return "Game name not yet generated."
#     video_bytes = open(video_file_path, 'rb').read()

#     response = self.generate_content_using_gemini(
#         model="gemini-2.5-pro",
#         contents=types.Content(
#             parts=[
#                 types.Part(
#                     inline_data=types.Blob(data=video_bytes, mime_type='video/mp4')
#                 ),
#                 types.Part(
#                     text=RANDOM_GAME_NAME_PROMPT
#                 )
#             ]
#         )
#     )

#     game_name = response.text.strip()
#     self.track_tokens(
#         response.usage_metadata.total_token_count, response.text
#     )
#     logger.info(f"Generated game name: {game_name}")
#     return game_name

# eval_data = eval_data_list[2]
# game_name = generate_game_name_from_video(
#     play_zero_agent,
#     eval_data.video_path,
# )
# print(f"Generated game name: {game_name}")

Generated game name: The player pushes a block to carefully transfer its fragile cap onto a target location without letting it fall.


In [3]:
# run_result = run_results[0]
# # generate goal and evaluate it
# goal = play_zero_agent.generate_top_hypothesis(
#     multiple_hypothesis_text=run_result["result"]["multiple_hypothesis_text"],
#     logical_analysis_actions_summary=run_result["result"]["logical_analysis_actions_summary"],
# )
# print_wrapped_text(f"Generated Goal:\n\n {goal}")

# eval_goal_relevance_result = eval_goal_relevance(
#     generated_goal=goal,
#     expected_goal=run["expected_goal"]
# )
# write_eval_log(eval_goal_relevance_result)

# print_wrapped_text(f"Eval Goal Relevance Result:\n\n {eval_goal_relevance_result}")

In [4]:
# from agents.templates.play_zero_agent import TOP_HYPOTHESIS_RETRIEVER_PROMPT


# prompt = TOP_HYPOTHESIS_RETRIEVER_PROMPT.format(
#     multiple_hypothesis_text=run_result["result"]["multiple_hypothesis_text"],
#     logical_analysis_actions_summary=run_result["result"]["logical_analysis_actions_summary"],
# )
# print_wrapped_text(f"Top Hypothesis Retriever Prompt:\n\n {prompt}")

In [ ]:
from google.genai import types

FRAME_EVENT_PROMPT = """- The action {action_taken} (description- {notes}) is made"""
EVENT_CHAIN_FILLER_MODEL = "gemini-2.5-flash"
EVENT_CHAIN_FILLER_PROMPT = """This video is game play with the below actions (WASD and click) taken on unknown game.

The game is designed based on below Constraints
- Easy for humans (can pick it up in <1 min of game play)
- Core Knowledge Priors (no language, trivia, cultural symbols)
- Should require no instructions to play
- Should be fun for humans and playable in 5-10 minutes
- Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)

Can you name this game?
"""
logger = logging.getLogger(__name__)
def generate_event_chain(self: PlayZeroAgent, effective_frames: list[FrameData]) -> str:
    count = 0
    event_chain = []
    prev_frame = effective_frames[0]
    for frame in effective_frames:
        action_text = frame.action_input.reasoning["previous_action_text"]
        frame_count = len(frame.frame)
        game_action = self.convert_action_text_to_game_action(action_text)
        notes = ""
        if game_action.is_complex():
            x = game_action.action_data.x
            y = game_action.action_data.y
        
            cell_value = prev_frame.frame[-1][y][x]
            color = self.get_color_for_cell_value(cell_value)
            if color:
                notes = f"Clicking {color} cell. But need to analyze the frames better"
            else:
                notes = "Need to analyse the frames better"
        frame_event = FRAME_EVENT_PROMPT.format(start_frame=count, end_frame=count + frame_count, action_taken=action_text, notes=notes)
        event_chain.append(frame_event)
        count += frame_count
    return "".join(event_chain)


def fill_missing_data_in_event_chain_using_video(
    self,
    video_file_path: str,
    event_chain_text: str,
    logical_analysis_actions_summary: str,
) -> str:
        """Filling event chain from a video file."""
        logger.info(f"Filling event chain from video: {video_file_path}")
        if not os.path.exists(video_file_path):
            logger.error(f"Video file does not exist: {video_file_path}")
            return "Analysis not yet done."
        video_bytes = open(video_file_path, 'rb').read()

        event_chain_text_response = self.generate_content_using_gemini(
            model=EVENT_CHAIN_FILLER_MODEL,
            contents=types.Content(
                parts=[
                    types.Part(
                        inline_data=types.Blob(data=video_bytes, mime_type='video/mp4')
                    ),
                    types.Part(text=EVENT_CHAIN_FILLER_PROMPT.format(
                        event_chain_text=event_chain_text,
                        logical_analysis_actions_summary=logical_analysis_actions_summary,
                    ))
                ]
            )
        )
        
        filled_event_chain_text = event_chain_text_response.text.strip()
        self.track_tokens(
            event_chain_text_response.usage_metadata.total_token_count, event_chain_text_response.text
        )
        logger.info(f"Event Chain filled: {filled_event_chain_text}")
        return filled_event_chain_text

frames = get_frames(runs[3]["scorecard_file_path"])
effective_frames = play_zero_agent.generate_video_from_grids(frames, "output.mp4", fps=1, skip_repeated_frames=True)
# event_chain_text = generate_event_chain(play_zero_agent, effective_frames=effective_frames)
event_chain_text = ""
logical_analysis_actions_summary = play_zero_agent.generate_logical_analysis_summary(frames)
filled_event_chain_text = fill_missing_data_in_event_chain_using_video(play_zero_agent, "output.mp4", event_chain_text, logical_analysis_actions_summary)
# print_wrapped_text(f"Event Chain:\n\n {event_chain_text}", width=90)
print(f"Filled Event Chain:\n\n {filled_event_chain_text}")

# elements_text = play_zero_agent.generate_element_titles_from_video(
#      video_file_path="output.mp4",
# )

# random_hypothesis = play_zero_agent.generate_multiple_random_hypothesis_from_video(
#      video_file_path="output.mp4",
#      logical_analysis_actions_summary=logical_analysis_actions_summary,
#      elements_text=elements_text,
# )
# print_wrapped_text(f"Random Hypothesis:\n\n {random_hypothesis}", width=90)


Filled Event Chain:

 Based on the visual elements and the inferred mechanics from the gameplay video, especially considering the constraints provided:

The game appears to be a minimalist puzzle game where the objective is to **activate** various nodes (represented by yellow squares and segments within black columns) by channeling the correct colored "flow" (from the blue and red sources) through "conduits" (the black columns). The top bar likely indicates progress or a timer for the current level. The subtle ghosting effects suggest the path of activation. The transition revealing distinct blue and red-linked columns implies a core mechanic of matching input colors to specific conduit types for activation.

Given these observations and the design constraints (easy to pick up, no instructions, core knowledge priors), a suitable name for this game would be:

**Conduit Flow**

Other strong candidates could be:
*   **Color Conduit**
*   **Node Flow**
*   **Circuit Stream**
Random Hypothe

In [11]:
print(f"Elements Text:\n\n {elements_text}")

Elements Text:

 Elements:
*   `Green_Top_Bar`: A thin, static green horizontal bar located at the very top of the screen.
*   `Gray_Upper_Background_Area`: A large, static gray rectangular section occupying the upper part of the screen.
*   `White_Lower_Background_Area`: A large, static white rectangular section occupying the lower part of the screen.
*   `Yellow_Player_Indicator_Dot`: A small square, initially yellow, that moves horizontally along the `Green_Top_Bar`. (It briefly turns green upon action).
*   `White_Progress_Squares`: A row of small, static white square outlines positioned horizontally in the `Gray_Upper_Background_Area`, which fill with color as progress is made.
*   `Tall_Black_Vertical_Pillar`: A prominent, tall black rectangular block that acts as a central structure or obstacle in the game. (Two appear during the video).
*   `Yellow_Pillar_Internal_Block`: A smaller, yellow square block embedded within the `Tall_Black_Vertical_Pillar` (visible from 0:00 to 0:03)

In [12]:
print(random_hypothesis)

Here are 5 hypotheses to explore the game:

1.  **Objective: Fill Progress Indicators:** The primary objective of the game is to fill all the `White_Progress_Squares` located in the `Gray_Upper_Background_Area` to achieve completion.
2.  **Core Mechanic: Color Transformation:** Advancing in the game requires the player to change the color of the `Yellow_Pillar_Internal_Block` and `Yellow_Ground_Block` to green, making them `Green_Pillar_Internal_Block` and `Green_Ground_Block`, respectively.
3.  **Player Action: Precise Alignment and Click:** The player controls the horizontal movement of the `Yellow_Player_Indicator_Dot` using WASD, aiming to align it with the `Tall_Black_Vertical_Pillar`. A `CLICK` action, performed when the `Yellow_Player_Indicator_Dot` is correctly positioned, triggers the color transformation of the pillar's internal block and the ground block, with the `Fading_Gray_Rectangle_Effect` indicating a successful interaction.
4.  **Progression: New Obstacles and Visual 